In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import streamlit as stl
import re

# -----------------------------

from catboost import CatBoostRegressor
from catboost import CatBoostClassifier

In [83]:
data = pd.read_excel('car details v4.xlsx')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [133]:
# data.info()
# data.columns
# data.shape
# data.isnull().sum()
# data['Seller Type'].value_counts().isnull().sum()
# data['Seller Type'].value_counts()

In [84]:
data.head()

,Make,Model,Price,Year,Kilometer,Fuel Type,Transmission,Location,Color,Owner,Seller Type,Engine,Max Power,Max Torque,Drivetrain,Length,Width,Height,Seating Capacity,Fuel Tank Capacity
0,Honda,Amaze 1.2 VX i-VTEC,505000,2017,87150,Petrol,Manual,Pune,Grey,First,Corporate,1198 cc,87 bhp @ 6000 rpm,109 Nm @ 4500 rpm,FWD,3990.0,1680.0,1505.0,5.0,35.0
1,Maruti Suzuki,Swift DZire VDI,450000,2014,75000,Diesel,Manual,Ludhiana,White,Second,Individual,1248 cc,74 bhp @ 4000 rpm,190 Nm @ 2000 rpm,FWD,3995.0,1695.0,1555.0,5.0,42.0
2,Hyundai,i10 Magna 1.2 Kappa2,220000,2011,67000,Petrol,Manual,Lucknow,Maroon,First,Individual,1197 cc,79 bhp @ 6000 rpm,112.7619 Nm @ 4000 rpm,FWD,3585.0,1595.0,1550.0,5.0,35.0
3,Toyota,Glanza G,799000,2019,37500,Petrol,Manual,Mangalore,Red,First,Individual,1197 cc,82 bhp @ 6000 rpm,113 Nm @ 4200 rpm,FWD,3995.0,1745.0,1510.0,5.0,37.0
4,Toyota,Innova 2.4 VX 7 STR [2016-2020],1950000,2018,69000,Diesel,Manual,Mumbai,Grey,First,Individual,2393 cc,148 bhp @ 3400 rpm,343 Nm @ 1400 rpm,RWD,4735.0,1830.0,1795.0,7.0,55.0


In [98]:
data['Engine_Num'] = data['Engine'].str.extract(r'(\d+\.?\d*)').astype(float)
data['Max_Power_Num'] = data['Max Power'].str.extract(r'(\d+\.?\d*)').astype(float)
data['Max_Torque_Num'] = data['Max Torque'].str.extract(r'(\d+\.?\d*)').astype(float)

data['Power RPM'] = data['Max Power'].str.split('@').str[1]
data['Power RPM'] = data['Power RPM'].str.replace('rpm','')

data['Torque RPM'] = data['Max Torque'].str.split('@').str[1]
data['Torque RPM'] = data['Torque RPM'].str.replace('rpm','')

cols = ['Engine_Num', 'Max_Power_Num', 'Max_Torque_Num', 'Torque RPM', 'Power RPM']

for col in cols:
    data[col] = data[col].astype(str).str.strip()
    data[col] = pd.to_numeric(data[col], errors='coerce')

In [99]:
train = data[data['Engine_Num'].notna()]
test = data[data['Engine_Num'].isna()]

x_train = train[['Model','Fuel Type']]
y_train = train['Engine_Num']
x_test = test[['Model','Fuel Type']]

model = CatBoostRegressor(verbose=False)
model.fit(x_train,y_train,cat_features=['Model','Fuel Type'])
pred = model.predict(x_test)
data.loc[data['Engine_Num'].isna(),'Engine_Num']= pred

data['Engine_Num'] = data['Engine_Num'].astype(int)

# ------------------------------------------------------------

train = data[data['Max_Power_Num'].notna()]
test = data[data['Max_Power_Num'].isna()]

x_train = train[['Model','Fuel Type']]
y_train = train['Max_Power_Num']
x_test = test[['Model','Fuel Type']]

model = CatBoostRegressor(verbose=False)
model.fit(x_train,y_train,cat_features=['Model','Fuel Type'])
pred = model.predict(x_test)
data.loc[data['Max_Power_Num'].isna(),'Max_Power_Num']= pred

data['Max_Power_Num'] = data['Max_Power_Num'].astype(int)

# ------------------------------------------------------------

train = data[data['Max_Torque_Num'].notna()]
test = data[data['Max_Torque_Num'].isna()]

x_train = train[['Model','Fuel Type']]
y_train = train['Max_Torque_Num']
x_test = test[['Model','Fuel Type']]

model = CatBoostRegressor(verbose=False)
model.fit(x_train,y_train,cat_features=['Model','Fuel Type'])
pred = model.predict(x_test)
data.loc[data['Max_Torque_Num'].isna(),'Max_Torque_Num']= pred

data['Max_Torque_Num'] = data['Max_Torque_Num'].astype(int)

# ------------------------------------------------------------

train = data[data['Torque RPM'].notna()]
test = data[data['Torque RPM'].isna()]

x_train = train[['Model']]
y_train = train['Torque RPM']
x_test = test[['Model']]

model = CatBoostRegressor(verbose=False)
model.fit(x_train,y_train,cat_features=['Model'])
pred = model.predict(x_test)
data.loc[data['Torque RPM'].isna(),'Torque RPM']= pred

# ------------------------------------------------------------

train = data[data['Power RPM'].notna()]
test = data[data['Power RPM'].isna()]

x_train = train[['Model']]
y_train = train['Power RPM']
x_test = test[['Model']]

model = CatBoostRegressor(verbose=False)
model.fit(x_train,y_train,cat_features=['Model'])
pred = model.predict(x_test)
data.loc[data['Power RPM'].isna(),'Power RPM']= pred

In [100]:
train = data[data['Drivetrain'].notna()]
test = data[data['Drivetrain'].isna()]

x_train = train[['Model']]
y_train = train['Drivetrain']
x_test = test[['Model']]

model = CatBoostClassifier(verbose=False)
model.fit(x_train,y_train,cat_features=['Model'])
pred = model.predict(x_test)
data.loc[data['Drivetrain'].isna(),'Drivetrain']= pred

In [ ]:
train = data[data['Length'].notna()]
test = data[data['Length'].isna()]

x_train = train[['Model']]
y_train = train['Length']
x_test = test[['Model']]

model = CatBoostRegressor(verbose=False)
model.fit(x_train,y_train,cat_features=['Model'])
pred = model.predict(x_test)
data.loc[data['Length'].isna(),'Length']= pred

data['Length'] = data['Length'].astype(int)

data['Length'] = data['Length'].astype(str) +' mm'

# ------------------------------------------------------------

train = data[data['Width'].notna()]
test = data[data['Width'].isna()]

x_train = train[['Model']]
y_train = train['Width']
x_test = test[['Model']]

model = CatBoostRegressor(verbose=False)
model.fit(x_train,y_train,cat_features=['Model'])
pred = model.predict(x_test)
data.loc[data['Width'].isna(),'Width']= pred

data['Width'] = data['Width'].astype(int)

data['Width'] = data['Width'].astype(str) +' mm'

# ------------------------------------------------------------

train = data[data['Height'].notna()]
test = data[data['Height'].isna()]

x_train = train[['Model']]
y_train = train['Height']
x_test = test[['Model']]

model = CatBoostRegressor(verbose=False)
model.fit(x_train,y_train,cat_features=['Model'])
pred = model.predict(x_test)
data.loc[data['Height'].isna(),'Height']= pred

data['Height'] = data['Height'].astype(int)

data['Height'] = data['Height'].astype(str) +' mm'

In [102]:
train = data[data['Seating Capacity'].notna()]
test = data[data['Seating Capacity'].isna()]

x_train = train[['Model']]
y_train = train['Seating Capacity']
x_test = test[['Model']]

model = CatBoostRegressor(verbose=False)
model.fit(x_train,y_train,cat_features=['Model'])
pred = model.predict(x_test)
data.loc[data['Seating Capacity'].isna(),'Seating Capacity']= pred

data['Seating Capacity'] = data['Seating Capacity'].astype(int)

In [103]:
train = data[data['Fuel Tank Capacity'].notna()]
test = data[data['Fuel Tank Capacity'].isna()]

x_train = train[['Model']]
y_train = train['Fuel Tank Capacity']
x_test = test[['Model']]

model = CatBoostRegressor(verbose=False)
model.fit(x_train,y_train,cat_features=['Model'])
pred = model.predict(x_test)
data.loc[data['Fuel Tank Capacity'].isna(),'Fuel Tank Capacity']= pred

data['Fuel Tank Capacity'] = data['Fuel Tank Capacity'].astype(int)

data['Fuel Tank Capacity'] = data['Fuel Tank Capacity'].astype(str) +' L'

In [106]:
cols2 = ['Engine_Num', 'Max_Power_Num', 'Max_Torque_Num', 'Torque RPM', 'Power RPM']

for col2 in cols2:
    data[col2] = data[col2].astype(str).str.strip()
    data[col2] = pd.to_numeric(data[col2], errors='coerce').astype(int)

In [ ]:
data['Engine_Num'] = data['Engine_Num'].astype(str) +' cc'
data['Kilometer'] = data['Kilometer'].astype(str) +' km'
data['Max_Power_Num'] = data['Max_Power_Num'].astype(str) +' bhp @ ' + data['Torque RPM'].astype(str) + ' rpm'
data['Max_Torque_Num'] = data['Max_Torque_Num'].astype(str) +' Nm @ ' + data['Power RPM'].astype(str) + ' rpm'

In [ ]:
data = data.drop(columns=['Engine','Max Power','Max Torque','Power RPM','Torque RPM'])

data.rename(columns={'Engine_Num':'Engine'}, inplace = True)
data.rename(columns={'Max_Power_Num':'Max Power'}, inplace = True)
data.rename(columns={'Max_Torque_Num':'Max Torque'}, inplace = True)
data.rename(columns={'Kilometer': 'Kilometers Driven'}, inplace = True)
data.rename(columns={'Make':'Brand'}, inplace = True)

In [118]:
cl = data.pop('Engine_Num')
data.insert(11, 'Engine_Num', cl)

cl = data.pop('Max_Power_Num')
data.insert(12, 'Max_Power_Num', cl)

cl = data.pop('Max_Torque_Num')
data.insert(13, 'Max_Torque_Num', cl)

cl = data.pop('Fuel Tank Capacity')
data.insert(18, 'Fuel Tank Capacity', cl)

In [4]:
data2 = pd.read_excel('car_details_cleaned.xlsx')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [86]:
data2.head()

,Brand,Model,Price,Year,Kilometers Driven,Fuel Type,Transmission,Location,Color,Owner,Seller Type,Engine,Power,Torque,Drivetrain,Length,Width,Height,Fuel Tank Capacity,Seating Capacity
0,Honda,Amaze 1.2 VX i-VTEC,505000,2017,87150 km,Petrol,Manual,Pune,Grey,First,Corporate,1198 cc,87 bhp @ 6000 rpm,109 Nm @ 4500 rpm,FWD,3990 mm,1680 mm,1505 mm,35 L,5
1,Maruti Suzuki,Swift DZire VDI,450000,2014,75000 km,Diesel,Manual,Ludhiana,White,Second,Individual,1248 cc,74 bhp @ 4000 rpm,190 Nm @ 2000 rpm,FWD,3995 mm,1695 mm,1555 mm,42 L,5
2,Hyundai,i10 Magna 1.2 Kappa2,220000,2011,67000 km,Petrol,Manual,Lucknow,Maroon,First,Individual,1197 cc,79 bhp @ 6000 rpm,112 Nm @ 4000 rpm,FWD,3585 mm,1595 mm,1550 mm,35 L,5
3,Toyota,Glanza G,799000,2019,37500 km,Petrol,Manual,Mangalore,Red,First,Individual,1197 cc,82 bhp @ 6000 rpm,113 Nm @ 4200 rpm,FWD,3995 mm,1745 mm,1510 mm,37 L,5
4,Toyota,Innova 2.4 VX 7 STR [2016-2020],1950000,2018,69000 km,Diesel,Manual,Mumbai,Grey,First,Individual,2393 cc,148 bhp @ 3400 rpm,343 Nm @ 1400 rpm,RWD,4735 mm,1830 mm,1795 mm,55 L,7


In [ ]:
data2['Power_rpm'] = data2['Max Torque'].str.extract(r'(@\s*\d+\s*rpm)')[0]
data2['Torque_rpm'] = data2['Max Power'].str.extract(r'(@\s*\d+\s*rpm)')[0]

data2['Max Power'] = data2.apply(lambda row: row['Max Power'].replace(data2['Power_rpm'][row.name],data2['Torque_rpm'][row.name]), axis=1)
data2['Max Torque'] = data2.apply(lambda row: row['Max Torque'].replace(data2['Torque_rpm'][row.name],data2['Power_rpm'][row.name]), axis=1)

data2['Power_bhp'] = data2['Max Power'].str.extract(r'(\d+\s*bhp)')
data2['Torque_Nm'] = data2['Max Torque'].str.extract(r'(\d+\s*Nm)')

data2['Power'] = data2['Power_bhp']+[' ']+data2['Power_rpm']
data2['Torque'] = data2['Torque_Nm']+[' ']+data2['Torque_rpm']

data2 = data2.drop(columns=['Max Power','Max Torque','Power_rpm','Torque_rpm','Power_bhp','Torque_Nm'])

cl = data2.pop('Power')
data2.insert(12, 'Power', cl)

cl = data2.pop('Torque')
data2.insert(13, 'Torque', cl)


In [4]:
data2.to_excel('car_details_cleaned.xlsx', index=None)

In [6]:
models_list = [
    "5-Series 520d Sedan", "5-Series 525d Sedan", "Ecosport Titanium 1.5L TDCi",
    "Figo Duratorq Diesel Titanium 1.4", "Amaze 1.2 VX AT i-VTEC", "Brio E MT",
    "Brio S MT", "CR-V 2.4 AT", "City 1.5 S MT", "City 1.5 V AT", "City 1.5 V MT",
    "City ZX CVT Petrol", "City ZX Diesel", "Civic 1.8V MT", "Creta E Plus 1.6 Petrol",
    "Elite i20 Magna Executive 1.2", "Elite i20 Sportz 1.2", "i20 Active 1.2 SX",
    "i20 Asta 1.2", "i20 Magna 1.2", "i20 Sportz 1.4 CRDI", "Evoque HSE Dynamic",
    "TUV300 T10", "Alto LXi CNG", "Ritz Vdi BS-IV", "Swift DZire VXI", "Swift LXi",
    "Swift VDi", "Swift VXi", "Swift ZDi", "Swift ZXi", "CLA 200 CDI Sport",
    "GLA 200 Sport", "Corolla Altis 1.8 G", "Polo Comfortline 1.2L (P)",
    "Polo GT TSI", "Polo Highline1.2L (P)", "Vento Highline Petrol AT",
]

filtered = data2[data2["Model"].isin(models_list)].copy()

cols_to_fix = ["Engine", "Power", "Torque", "Length", "Width", "Height",
               "Fuel Tank Capacity", "Seating Capacity"]

filtered = filtered[["Model"] + cols_to_fix].copy()

for col in cols_to_fix:
    filtered[col] = filtered.groupby("Model")[col].transform(
        lambda x: x.mode().iloc[0]
    )

filtered = filtered.sort_values("Model")
filtered

,Model,Engine,Power,Torque,Length,Width,Height,Fuel Tank Capacity,Seating Capacity
334,5-Series 520d Sedan,1995 cc,184 bhp @ 4000 rpm,380 Nm @ 1750 rpm,4899 mm,2094 mm,1464 mm,62 L,5
180,5-Series 520d Sedan,1995 cc,184 bhp @ 4000 rpm,380 Nm @ 1750 rpm,4899 mm,2094 mm,1464 mm,62 L,5
134,5-Series 520d Sedan,1995 cc,184 bhp @ 4000 rpm,380 Nm @ 1750 rpm,4899 mm,2094 mm,1464 mm,62 L,5
900,5-Series 520d Sedan,1995 cc,184 bhp @ 4000 rpm,380 Nm @ 1750 rpm,4899 mm,2094 mm,1464 mm,62 L,5
868,5-Series 525d Sedan,1995 cc,217 bhp @ 4400 rpm,450 Nm @ 1500 rpm,4899 mm,2094 mm,1464 mm,62 L,5
276,5-Series 525d Sedan,1995 cc,217 bhp @ 4400 rpm,450 Nm @ 1500 rpm,4899 mm,2094 mm,1464 mm,62 L,5
1449,5-Series 525d Sedan,1995 cc,217 bhp @ 4400 rpm,450 Nm @ 1500 rpm,4899 mm,2094 mm,1464 mm,62 L,5
1505,Alto LXi CNG,796 cc,39 bhp @ 6200 rpm,54 Nm @ 3000 rpm,3495 mm,1475 mm,1460 mm,35 L,5
24,Alto LXi CNG,796 cc,39 bhp @ 6200 rpm,54 Nm @ 3000 rpm,3495 mm,1475 mm,1460 mm,35 L,5
1038,Alto LXi CNG,796 cc,39 bhp @ 6200 rpm,54 Nm @ 3000 rpm,3495 mm,1475 mm,1460 mm,35 L,5


In [ ]:
def fix_and_filter(input_path="car_details_cleaned.xlsx",
                    output_path="car_details_fixed.xlsx"):

    data2 = pd.read_excel(input_path)

    cols_to_fix = ["Engine", "Power", "Torque", "Length", "Width", "Height",
                   "Fuel Tank Capacity", "Seating Capacity"]

    for col in cols_to_fix:
        data2[col] = data2.groupby("Model")[col].transform(
            lambda x: x.mode().iloc[0]
        )

    data2.to_excel(output_path, index=False)
    print(f"saved{output_path}")

    models_list = [
        "5-Series 520d Sedan", "5-Series 525d Sedan", "Ecosport Titanium 1.5L TDCi",
        "Figo Duratorq Diesel Titanium 1.4", "Amaze 1.2 VX AT i-VTEC", "Brio E MT",
        "Brio S MT", "CR-V 2.4 AT", "City 1.5 S MT", "City 1.5 V AT", "City 1.5 V MT",
        "City ZX CVT Petrol", "City ZX Diesel", "Civic 1.8V MT", "Creta E Plus 1.6 Petrol",
        "Elite i20 Magna Executive 1.2", "Elite i20 Sportz 1.2", "i20 Active 1.2 SX",
        "i20 Asta 1.2", "i20 Magna 1.2", "i20 Sportz 1.4 CRDI", "Evoque HSE Dynamic",
        "TUV300 T10", "Alto LXi CNG", "Ritz Vdi BS-IV", "Swift DZire VXI", "Swift LXi",
        "Swift VDi", "Swift VXi", "Swift ZDi", "Swift ZXi", "CLA 200 CDI Sport",
        "GLA 200 Sport", "Corolla Altis 1.8 G", "Polo Comfortline 1.2L (P)",
        "Polo GT TSI", "Polo Highline1.2L (P)", "Vento Highline Petrol AT",
    ]

    filtered = data2[data2["Model"].isin(models_list)].copy()
    filtered = filtered[["Model"] + cols_to_fix].copy()
    filtered = filtered.sort_values("Model")

    return data2, filtered


data2, filtered = fix_and_filter()
filtered

In [ ]:
data2['Fuel Type'] = data2['Fuel Type'].replace('CNG + CNG', 'CNG')
data2['Owner'] = data2['Owner'].replace('4 or More', 'More Than 4')

In [10]:
data2.to_excel('car_details_fixed.xlsx', index=False)